# Voice to Text Sentiment Analysis (RNN/LSTM)
Speech emotion recognition using MFCC features + LSTM, mapped to sentiment (positive/negative/neutral).

Dataset: RAVDESS

In [ ]:
!pip install librosa tensorflow scikit-learn matplotlib -q

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

## Mount Drive and set dataset path
Upload the RAVDESS dataset to your Google Drive and update the path below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/RAVDESS"

In [ ]:
emotion_labels = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

def emotion_to_sentiment(emotion):
    if emotion in ["happy", "calm", "surprised"]:
        return "positive"
    elif emotion == "neutral":
        return "neutral"
    else:
        return "negative"

## Feature extraction (MFCC)

In [ ]:
def extract_mfcc(file_path, max_len=100):
    audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfcc = mfcc.T

    if mfcc.shape[0] < max_len:
        pad_width = max_len - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode='constant')
    else:
        mfcc = mfcc[:max_len, :]

    return mfcc

## Load dataset

In [ ]:
X = []
y = []

print("loading files...")

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)
            parts = file.split("-")
            emotion_code = parts[2]
            emotion = emotion_labels.get(emotion_code)

            if emotion is None:
                continue

            features = extract_mfcc(file_path)
            X.append(features)
            y.append(emotion)

X = np.array(X)
print(len(X))
print(X[0].shape)

## Encode labels

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

print(list(le.classes_))

## Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

## Build LSTM model

In [ ]:
model = Sequential()
model.add(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(64))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dense(y_categorical.shape[1], activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

## Train the model

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=40,
    batch_size=32
)

## Evaluate

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("accuracy:", accuracy)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig("training_graphs.png")
plt.show()

## Save the model

In [ ]:
model.save("speech_emotion_lstm_model.keras")

## Predict on a new audio file

In [ ]:
def predict_emotion(file_path):
    features = extract_mfcc(file_path)
    features = np.expand_dims(features, axis=0)

    prediction = model.predict(features)
    predicted_index = np.argmax(prediction)
    predicted_emotion = le.inverse_transform([predicted_index])[0]
    predicted_sentiment = emotion_to_sentiment(predicted_emotion)

    print("Predicted Emotion:", predicted_emotion)
    print("Predicted Sentiment:", predicted_sentiment)

    return predicted_emotion, predicted_sentiment

In [ ]:
# predict_emotion("test_audio.wav")